In [0]:
from pyspark.sql import functions as F
import mlflow

mlflow.set_registry_uri("databricks-uc")

SILVER = "workspace.s4lake_silver"
GOLD = "workspace.s4lake_gold"
DATA_CORTE = "2026-09-22"
N_DIAS_ALVO = 30
NOME_MODELO = f"{GOLD}.modelo_risco_atraso"

FEATURES_NUM = ["valor"]
FEATURES_HIST = ["atraso_medio_hist", "sem_historico"]
FEATURES_CAT = ["ramo_atividade", "uf", "mes_vencimento", "prazo_dias"]

In [0]:
def comentar_tabela(tabela, comentario_tabela, comentarios_colunas):
    nome = f"{GOLD}.{tabela}"

    def texto_sql(texto):
        # Escapa aspas simples para não fecharem a string do SQL antes da hora
        return texto.replace("'", "\\'")

    spark.sql(f"COMMENT ON TABLE {nome} IS '{texto_sql(comentario_tabela)}'")

    colunas_tabela = spark.table(nome).columns
    for coluna, texto in comentarios_colunas.items():
        if coluna in colunas_tabela:
            spark.sql(f"ALTER TABLE {nome} ALTER COLUMN {coluna} COMMENT '{texto_sql(texto)}'")

    sem_comentario = [c for c in colunas_tabela if c not in comentarios_colunas]
    nao_encontradas = [c for c in comentarios_colunas if c not in colunas_tabela]
    print(f"{tabela}: {len(colunas_tabela) - len(sem_comentario)} de {len(colunas_tabela)} colunas comentadas")
    print("  Colunas sem comentário:", sem_comentario)
    print("  Comentários sem coluna:", nao_encontradas)

def salvar(df, schema, tabela):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{schema}.{tabela}")

In [0]:
fato_titulos = spark.table(f"{GOLD}.fato_titulos")

# Abertos que ainda não completaram 30 dias após o vencimento: desfecho desconhecido
a_pontuar = (
    fato_titulos
    .withColumn("dias_desde_vencimento",
                F.datediff(F.lit(DATA_CORTE).cast("date"), F.col("data_vencimento")))
    .filter((F.col("status") == "aberto") & (F.col("dias_desde_vencimento") <= N_DIAS_ALVO))
)

print(a_pontuar.count())
a_pontuar.groupBy("faixa_aging").count().show()

In [0]:
clientes_dim = (
    spark.table(f"{SILVER}.clientes")
    .select("cod_cliente", "ramo_atividade", "ramo_descricao", "uf")
)
features_hist = spark.table(f"{GOLD}.features_historico_cliente")

dados_pontuar = (
    a_pontuar
    .join(clientes_dim, on="cod_cliente", how="left")
    .join(features_hist, on="cod_fatura", how="left")
    .withColumn("mes_vencimento", F.month("data_vencimento"))
    .select(
        "cod_fatura", "cod_cliente",                                       # chave
        "data_vencimento", "situacao", "faixa_aging", "ramo_descricao",   # contexto para o Power BI
        "valor", "prazo_dias", "mes_vencimento", "ramo_atividade", "uf",  # features do título
        "atraso_medio_hist",                                               # feature de histórico
    )
)

print(dados_pontuar.count())
dados_pontuar.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in dados_pontuar.columns]
).show()

In [0]:
dados_pontuar_pd = dados_pontuar.toPandas()
dados_pontuar_pd["valor"] = dados_pontuar_pd["valor"].astype(float)
dados_pontuar_pd["sem_historico"] = dados_pontuar_pd["atraso_medio_hist"].isna().astype(int)

X_pontuar = dados_pontuar_pd[FEATURES_NUM + FEATURES_HIST + FEATURES_CAT]

modelo = mlflow.sklearn.load_model(f"models:/{NOME_MODELO}@campeao")
dados_pontuar_pd["probabilidade_atraso"] = modelo.predict_proba(X_pontuar)[:, 1]
dados_pontuar_pd["valor_em_risco"] = dados_pontuar_pd["valor"] * dados_pontuar_pd["probabilidade_atraso"]

print(X_pontuar.shape)
print(dados_pontuar_pd["probabilidade_atraso"].describe())
print("valor em risco:", round(dados_pontuar_pd["valor_em_risco"].sum(), 2))

In [0]:
risco_titulos = (
    spark.createDataFrame(dados_pontuar_pd.drop(columns=["sem_historico"]))
    .withColumn("valor", F.col("valor").cast("decimal(15,2)"))
    .withColumn("probabilidade_atraso", F.round("probabilidade_atraso", 4))
    .withColumn("valor_em_risco", F.col("valor_em_risco").cast("decimal(15,2)"))
    .withColumn("data_corte", F.lit(DATA_CORTE).cast("date"))
)

risco_titulos.printSchema()

In [0]:
salvar(risco_titulos, GOLD, "risco_titulos")

comentar_tabela(
    "risco_titulos",
    "Risco previsto de atraso grave (mais de 30 dias ou não pagamento) dos títulos em aberto sem desfecho "
    "conhecido na data de corte. Uma linha por título (cod_fatura). Pontuado pelo modelo registrado como "
    "modelo_risco_atraso@campeao. Base da lista de cobrança preventiva.",
    {
        "cod_fatura": "Número da fatura que originou o título; chave de junção com a fato_titulos.",
        "cod_cliente": "Código do cliente devedor.",
        "data_vencimento": "Data de vencimento do título.",
        "situacao": "Situação na data de corte: a vencer ou vencido (até 30 dias).",
        "faixa_aging": "Faixa de atraso na data de corte: a vencer ou 1-30.",
        "ramo_descricao": "Ramo de atividade do cliente, por extenso.",
        "valor": "Valor bruto do título, em R$.",
        "prazo_dias": "Prazo de pagamento concedido no título, em dias. Feature do modelo.",
        "mes_vencimento": "Mês do vencimento (1 a 12). Feature do modelo.",
        "ramo_atividade": "Código do ramo de atividade do cliente. Feature do modelo.",
        "uf": "UF do cliente. Feature do modelo.",
        "atraso_medio_hist": "Atraso médio point-in-time do cliente na emissão, em dias. Feature do modelo; nulo sem histórico.",
        "probabilidade_atraso": "Probabilidade prevista de atraso grave, de 0 a 1. Use para ordenar a cobrança; "
                                "na validação, o modelo superestimou a taxa nos extremos.",
        "valor_em_risco": "Valor x probabilidade_atraso, em R$. Soma é a perda esperada; tende a superestimar.",
        "data_corte": "Data de corte da pontuação, contra a qual os títulos foram classificados.",
    },
)

In [0]:
%sql
SELECT
  COUNT(*) AS linhas,
  COUNT(DISTINCT cod_fatura) AS chaves_unicas,
  SUM(CASE WHEN probabilidade_atraso IS NULL THEN 1 ELSE 0 END) AS prob_nulas,
  MIN(probabilidade_atraso) AS prob_min,
  MAX(probabilidade_atraso) AS prob_max,
  SUM(valor) AS valor_total,
  SUM(valor_em_risco) AS valor_em_risco
FROM workspace.s4lake_gold.risco_titulos